# NB10 — Secondary analyses: reverse transport, time vs place, subgroups

**Purpose.** (a) Reverse direction: the eICU-2014 LightGBM applied to MIMIC-IV. (b) Time vs place: drift of the MIMIC model across MIMIC calendar periods compared with the drift from MIMIC to eICU. (c) Fairness: calibration by age band, sex and race before and after recalibration.

**Research question.** Secondary

| | |
|---|---|
| **Inputs** | models + `pred_eval_strategies.parquet` |
| **Outputs** | Tables S1–S3, Figure S1 |
| **Free-Colab runtime** | 10–15 min |

**How to run:** `Runtime → Run all`. If Colab disconnects, just run all again — finished steps are cached.
At the end, download the zip printed in the last cell into your local `Plan A/v4/results/` folder.

In [ ]:
# ==== Setup (identical in every notebook) ====
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ.setdefault('TRANSPORT_BASE', '/content/drive/MyDrive/Plan A/v4')
BASE = os.environ['TRANSPORT_BASE']
sys.path.insert(0, os.path.join(BASE, 'code'))
import importlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import transport_utils as tu
importlib.reload(tu)
warnings.filterwarnings('ignore')
tu.set_style()
OUT = tu.init_notebook('NB10')

In [ ]:
import joblib
m, e = tu.load_analysis()
FEATS = tu.feature_set('V1')
pm = pd.read_parquet(os.path.join(tu.MODELS, 'pred_mimic.parquet'))
pe = pd.read_parquet(os.path.join(tu.MODELS, 'pred_eicu.parquet'))
evs = pd.read_parquet(os.path.join(tu.MODELS, 'pred_eval_strategies.parquet'))
refit = joblib.load(os.path.join(tu.MODELS, 'lgbm_eicu2014_refit.pkl'))
B = tu.n_boot(500)
MET = ['auroc', 'oe', 'citl', 'slope', 'ici']
def row(label, y, p, clusters=None):
    ci, _ = tu.bootstrap(y, p, clusters=clusters, B=B, metrics=MET)
    return {r['metric']: tu.fmt_ci(r['estimate'], r['lo'], r['hi']) for _, r in ci.iterrows()} | {'setting': label, 'n': len(y)}

### (a) Reverse transport and (b) time vs place

In [ ]:
p_rev = refit.predict_proba(m[FEATS].astype(float))[:, 1]
ev_ = pe[pe['split'] == 'eval']
rows = [
    row('MIMIC -> MIMIC internal (OOF, 2008-16)', pm.query("set=='dev_oof'")['mortality'].values, pm.query("set=='dev_oof'")['p_lgbm'].values),
]
for g, d in pm[pm['set'] == 'temporal_test'].groupby('anchor_year_group'):
    rows.append(row(f'MIMIC -> MIMIC later period {g} (time)', d['mortality'].values, d['p_lgbm'].values))
rows += [
    row('MIMIC -> eICU 2015 (place)', ev_['mortality'].values, ev_['p_lgbm'].values, ev_['hospital_id'].values),
    row('eICU 2014 -> eICU 2015 (same network, next year)', evs['mortality'].values, evs['p_refit'].values, evs['hospital_id'].values),
    row('eICU 2014 -> MIMIC all (reverse place)', m['mortality'].values, p_rev),
]
tp = pd.DataFrame(rows)[['setting', 'n'] + MET]
tu.save_table(tp, 'tableS1_time_vs_place_reverse')

### (c) Subgroup calibration before and after recalibration (eICU 2015)

In [ ]:
evs['age_band'] = pd.cut(evs['age'], [17, 49, 64, 79, 200], labels=['18-49', '50-64', '65-79', '80+'])
evs['sex'] = evs['sex_male'].map({1.0: 'Male', 0.0: 'Female'})
sg = []
for var in ['age_band', 'sex', 'race_group']:
    for lev, d in evs.groupby(var, observed=True):
        if d['mortality'].sum() < 20:
            continue
        for s in ['S0_none', 'S1b_global_logistic', 'S3_eb']:
            r = tu.evaluate(d['mortality'].values, d[s].values, full=False)
            sg.append(dict(variable=var, level=str(lev), strategy=tu.STRATEGY_LABELS[s], n=r['n'],
                           events=r['events'], auroc=r['auroc'], oe=r['oe'], slope=r['slope']))
sg = pd.DataFrame(sg)
tu.save_table(sg.round(3), 'tableS2_subgroups')
fig, ax = plt.subplots(figsize=(8, 4.5))
lv = sg[['variable', 'level']].drop_duplicates().reset_index(drop=True)
for i, s in enumerate(sg['strategy'].unique()):
    d = sg[sg['strategy'] == s].merge(lv.reset_index(), on=['variable', 'level'])
    ax.scatter(d['oe'], d['index'] + (i - 1) * 0.2, color=tu.SERIES[i], s=22, label=s)
ax.set_yticks(range(len(lv))); ax.set_yticklabels(lv['variable'] + ': ' + lv['level'])
ax.axvspan(*tu.OE_ACCEPT, color=tu.C['aqua'], alpha=0.1); ax.axvline(1, color=tu.C['ink2'], lw=1)
ax.set_xlabel('O/E'); ax.set_title('Subgroup calibration, eICU 2015'); ax.legend(loc='lower right')
tu.save_fig(fig, 'figS1_subgroups'); plt.show()

In [ ]:
tu.finish_notebook({'time_vs_place_reverse': tp.to_dict('records'), 'subgroups': sg.round(4).to_dict('records')})